# Distributions and Shape

**DS4DH · Module 02 — Describing and Comparing Data**

*Technique:* Histograms, skewness, and why shape decides the summary statistic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/02b_distributions_shape.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

The previous notebook chose between mean and median using one derived number.
This one looks at the thing itself.

A summary statistic is a compression. A histogram is what you compress *from*,
and it is the only way to see multi-modality, boundary effects, and outliers —
none of which appear in a table of means.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
d = base.dropna(subset=['Renter'])

fig, ax = plt.subplots()
ax.hist(d['Renter'], bins=30, edgecolor='white', linewidth=0.6)
ax.axvline(d['Renter'].mean(), color='#E8663D', lw=2,
           label=f'mean {d["Renter"].mean():.1f}')
ax.axvline(d['Renter'].median(), color='#3DA5D9', lw=2, ls='--',
           label=f'median {d["Renter"].median():.1f}')
ax.axvline(30, color='#888', lw=1.5, ls=':', label='30% affordability line')
ax.set_xlabel('Renter STIR (%)')
ax.set_ylabel('CSDs')
ax.set_title(f'Renter housing burden across {len(d)} CSDs')
ax.legend()
plt.show()

Two things to read off that chart:

- the **mean sits to the right of the median** — the right tail is pulling it
- a visible group of CSDs sits **above the 30% line**, the conventional threshold
  at which housing is called unaffordable

In [ ]:
# Quantify the shape.
s = d['Renter']
print(f'n           {len(s)}')
print(f'skewness    {stats.skew(s):>7.3f}   (0 = symmetric, >0 = right tail)')
print(f'kurtosis    {stats.kurtosis(s):>7.3f}   (0 = normal-tailed, >0 = heavy)')
print()
print(f'above 30%:  {(s > 30).sum()} of {len(s)} CSDs ({(s > 30).mean():.1%})')

### 🔧 Your turn 1

Change `bins=30` to `bins=8`, then to `bins=60`, re-running each time.

At which bin count does the second cluster above 30% disappear? Bin width is an
analyst's choice that changes what a reader sees — this is the same problem the
KDE bandwidth poses in notebook 10a.

## Shape differs by city

A pooled histogram hides that these are four housing markets, not one.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, city in zip(axes.ravel(), CITIES):
    s = d[d['cma'] == city]['Renter']
    ax.hist(s, bins=15, edgecolor='white', linewidth=0.5)
    ax.axvline(s.median(), color='#3DA5D9', lw=2, ls='--')
    ax.axvline(30, color='#888', lw=1, ls=':')
    ax.set_title(f'{city}  (n={len(s)}, median {s.median():.1f})')
    ax.set_xlabel('Renter STIR (%)')
plt.tight_layout()
plt.show()

In [ ]:
# Same axis for all four, so the panels are comparable — see notebook 10c.
print(f'{"City":<12}{"n":>5}{"skew":>8}{"median":>9}{"% over 30":>11}')
print('-' * 45)
for city in CITIES:
    s = d[d['cma'] == city]['Renter']
    sk = stats.skew(s) if len(s) > 2 else float('nan')
    print(f'{city:<12}{len(s):>5}{sk:>8.2f}{s.median():>9.1f}{(s > 30).mean():>10.1%}')

### 🔧 Your turn 2

One city has far fewer CSDs than the others. Look at its histogram.

Would you report a skewness figure for it? At what sample size does a shape
statistic stop being meaningful?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** At 8 bins the cluster above 30% merges into the main body and
disappears. At 60 bins it fragments into noise. Neither is wrong; both are
choices. The honest practice is to look at several and pick one that neither
invents structure nor conceals it — and to say in the caption what you chose.

**Your turn 2.** Toronto and Edmonton have around 22–23 CSDs each. Skewness is a
third-moment statistic and is very unstable below roughly 30 observations — a
single extreme value can flip its sign. Report the median and the range for these
cities and leave shape statistics to Montréal, which has 88. The general rule:
the more moments a statistic uses, the more data it needs before it means
anything.

</details>

## Where this stops

You can now see one variable's distribution and describe it honestly. Comparing
two distributions — and deciding whether a difference between them is real — is
the next two notebooks, and then all of Module 04.